# 📝 [제 14편] CNN - 합성곱으로 사진의 특징 찾기

**학년/반/번호:** 2학년 ( )반 ( )번  **이름:** (           )

> 근거 자료: 박성호, 「CNN (I) - 컨볼루션 연산 • 풀링 • 패딩」 (ML_LEC_32)

---

## 🎯 학습 목표
1. 합성곱(컨볼루션) 연산을 손으로 계산하고, 필터가 특징을 찾는 원리를 설명할 수 있다.
2. ReLU 와 풀링(pooling)의 역할을 알고 계산할 수 있다.
3. 패딩과 스트라이드에 따른 출력 크기를 공식으로 구할 수 있다.

---

## 🧭 지난 시간 복습
* 제 13편 도전 과제: 지금까지의 신경망은 28 × 28 사진을 **784 개짜리 한 줄로 펴서** 넣었습니다. → 픽셀끼리 **위아래·좌우로 이웃** 한다는 정보가 사라집니다.
* CNN (Convolutional Neural Network, 합성곱 신경망) 은 사진을 **2차원 그대로** 다루며 특징(feature)을 찾습니다.

[⬅ 이전 : 제 13편 오차역전파 (2) - 은닉층 공식과 MNIST 고속 학습](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/13_backprop_2.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/00_index.ipynb)　|　[다음 : 제 15편 RNN - 순서를 기억하는 신경망 ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/15_rnn.ipynb)

> 💡 **처음 여는 경우** : 파일 → **Drive에 사본 저장** 을 누르고, 내 사본에서 공부하세요. (빈칸을 채우고 저장할 수 있습니다)
>
> 🧹 **다 본 뒤에는** : 수정 → **모든 출력 지우기** 후 저장하면 애니메이션 때문에 무거워진 사본이 다시 가벼워집니다.

## ⚙️ 0단계. 준비하기
아래 셀을 **가장 먼저 한 번** 실행하세요. (그래프 한글 글꼴과 애니메이션 설정)

In [ ]:
#@title ▶ 준비 셀 (먼저 실행하세요)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib import animation
from IPython.display import HTML, display
import urllib.request, os

font_path = 'NanumGothic.ttf'
try:
    if not os.path.exists(font_path):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf', font_path)
    fm.fontManager.addfont(font_path)
    plt.rc('font', family=fm.FontProperties(fname=font_path).get_name())
except Exception as e:
    print('글꼴을 내려받지 못했습니다(그래프의 한글이 깨질 수 있음):', e)
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['animation.embed_limit'] = 50
def show_anim(anim):
    # 애니메이션을 코랩 화면에 재생 버튼과 함께 보여 주기
    display(HTML(anim.to_jshtml()))
np.set_printoptions(precision=6, suppress=True)
print('✅ 준비 완료!')

## 📖 개념 다지기 1. CNN 의 구조

```
 일반 신경망 :  입력 → [완전연결 + sigmoid] → [완전연결 + sigmoid] → 출력
 CNN       :  입력 → [conv → ReLU → pooling] → [conv → ReLU → pooling] → [완전연결] → 출력
```

| 층 | 하는 일 |
|:---|:---|
| **conv (합성곱)** | 필터(가중치 묶음)를 사진 위로 옮겨 가며 곱해 **특징(feature)** 을 뽑는다 |
| **ReLU** | 활성화 함수. 음수는 0, 양수는 그대로 → ReLU(x) = max(0, x) |
| **pooling (풀링)** | 작은 구역을 최댓값(max) 등 하나로 **압축** 해서 계산량을 줄인다 |

## 📖 개념 다지기 2. 합성곱(컨볼루션) 연산 ⭐

필터를 일정 간격(**스트라이드**)으로 옮기면서, 입력과 필터의 **같은 자리끼리 곱해 모두 더합니다.**

　입력 (4 × 4)　　　　　　 필터 (3 × 3)
　( 1　2　3　0 )　　　　 ( 2　0　1 )
　( 0　1　2　3 )　　　　 ( 0　1　2 )
　( 3　0　1　2 )　　　　 ( 1　0　2 )
　( 2　3　0　1 )

* 왼쪽 위 (1행 1열):  1×2 + 2×0 + 3×1 + 0×0 + 1×1 + 2×2 + 3×1 + 0×0 + 1×2 = **15**
* 오른쪽으로 1칸:　2×2 + 3×0 + 0×1 + 1×0 + 2×1 + 3×2 + 0×1 + 1×0 + 2×2 = ( 　　 )
* 아래로 1칸:　　0×2 + 1×0 + 2×1 + 3×0 + 0×1 + 1×2 + 2×1 + 3×0 + 0×2 = ( 　　 )
* 오른쪽 아래:　 ( 　　　　　　　　　　　　　　　　　　　　 ) = 15

결과 (2 × 2) 에 바이어스 3 을 더하면 **특징 맵 (feature map)** :  ( 18　19 ) / ( ( 　 )　18 )

In [ ]:
def conv2d(A, F, stride=1):
    H, W = A.shape; FH, FW = F.shape
    OH = (H - FH) // stride + 1; OW = (W - FW) // stride + 1
    out = np.zeros((OH, OW))
    for i in range(OH):
        for j in range(OW):
            window = A[i*stride:i*stride+FH, j*stride:j*stride+FW]   # 필터가 덮는 부분
            out[i, j] = np.sum(window * F)                           # 같은 자리끼리 곱해서 모두 더하기
    return out

A = np.array([[1, 2, 3, 0], [0, 1, 2, 3], [3, 0, 1, 2], [2, 3, 0, 1]])
F = np.array([[2, 0, 1], [0, 1, 2], [1, 0, 2]])
C = conv2d(A, F)
print('컨볼루션 결과 =\n', C)
print('바이어스 3 을 더한 특징 맵 =\n', C + 3)

## 🎬 애니메이션 1. 필터가 미끄러지며 계산하는 모습

In [ ]:
#@title 🎬 애니메이션 1 실행 (컨볼루션)
def draw_grid(ax, M, x0, y0, hl=None, hc='#ffe08a', title='', fmt='{:g}', hide=None):
    r, c = M.shape
    for i in range(r):
        for j in range(c):
            on = hl is not None and hl[0] <= i < hl[0] + hl[2] and hl[1] <= j < hl[1] + hl[3]
            ax.add_patch(plt.Rectangle((x0 + j, y0 - i), 1, 1, fc=hc if on else 'white', ec='k', lw=1.2))
            if hide is None or (i, j) in hide:
                ax.text(x0 + j + .5, y0 - i + .5, fmt.format(M[i, j]), ha='center', va='center', fontsize=14)
    ax.text(x0 + c/2, y0 + 1.3, title, ha='center', fontsize=13, weight='bold')

pos = [(0, 0), (0, 1), (1, 0), (1, 1)]
frames = [k for k in range(4) for _ in range(7)] + [4]*8
fig, ax = plt.subplots(figsize=(12, 4.6))
def draw(f):
    k = frames[f]
    ax.clear(); ax.set_xlim(-0.5, 17.5); ax.set_ylim(-4.6, 2.6); ax.axis('off'); ax.set_aspect('equal')
    if k < 4:
        i, j = pos[k]
        draw_grid(ax, A, 0, 1, hl=(i, j, 3, 3), title='입력 (4 × 4)')
        draw_grid(ax, F, 5.5, 1, hl=(0, 0, 3, 3), hc='#cfe3ff', title='필터 (3 × 3)')
        draw_grid(ax, C, 10, 1, hl=(i, j, 1, 1), hc='#b8f0b8', title='결과 (2 × 2)', hide=set(pos[:k+1]))
        terms = ' + '.join(f'{A[i+a, j+b]}×{F[a, b]}' for a in range(3) for b in range(3))
        ax.text(8.5, -3.9, f'{terms} = {int(C[i, j])}', ha='center', fontsize=11, color='darkgreen')
    else:
        draw_grid(ax, A, 0, 1, title='입력 (4 × 4)'); draw_grid(ax, F, 5.5, 1, hc='#cfe3ff', title='필터 (3 × 3)')
        draw_grid(ax, C, 10, 1, title='결과'); ax.text(12.6, 0.5, '+ 3 =', fontsize=15, va='center')
        draw_grid(ax, C + 3, 14.3, 1, hc='#b8f0b8', hl=(0, 0, 2, 2), title='특징 맵')
    ax.text(4.75, 0, '*', fontsize=26, ha='center', va='center'); ax.text(9.25, 0, '=', fontsize=22, ha='center', va='center')
anim = animation.FuncAnimation(fig, draw, frames=len(frames), interval=170)
plt.close(fig)
show_anim(anim)

## 📖 개념 다지기 3. ReLU 와 풀링

**ReLU(x) = max(0, x)** : 음수는 0 으로, 양수는 그대로.  예) ReLU(−3) = ( 　 ), ReLU(5) = ( 　 )

**풀링** : 구역을 대표값 하나로 압축
| 종류 | ( 15 16 / 6 15 ) 의 결과 |
|:-:|:-:|
| max pooling (최댓값) | 16 |
| min pooling (최솟값) | ( 　 ) |
| average pooling (평균) | ( 　 ) |

→ CNN 에서는 **max pooling** 을 가장 많이 씁니다. (가장 강한 특징만 남기기)

In [ ]:
M = np.array([[15, 16], [6, 15]])
print('max =', M.max(), ', min =', M.min(), ', average =', M.mean())

def max_pool(X, size=2):
    H, W = X.shape
    return X[:H//size*size, :W//size*size].reshape(H//size, size, W//size, size).max(axis=(1, 3))
X = np.array([[1, -2, 3, 0], [4, 5, -6, 1], [-1, 0, 2, 8], [3, -4, 7, 2]])
print('\nX =\n', X)
print('ReLU(X) =\n', np.maximum(0, X))
print('2 × 2 max pooling =\n', max_pool(np.maximum(0, X)))

> **[연습 1]** 위 X 에 ReLU 를 적용한 뒤 2 × 2 max pooling 한 결과를 손으로 먼저 구하세요.  ( 　 　 ) / ( 　 　 )

## 📖 개념 다지기 4. 패딩(padding)과 출력 크기

컨볼루션을 하면 크기가 줄어듭니다. (4 × 4 → 2 × 2) 이를 막으려고 입력 둘레를 0 으로 채우는 것이 **패딩** 입니다.

　패딩 1 : 4 × 4 입력 → 6 × 6 → 3 × 3 필터로 컨볼루션 → **4 × 4** (크기 유지!)

### 출력 크기 공식
입력 (H, W), 필터 (FH, FW), 패딩 P, 스트라이드 S 일 때

> **OH = (H + 2P − FH) / S + 1,　 OW = (W + 2P − FW) / S + 1**

| | 입력 | 필터 | 패딩 | 스트라이드 | 출력 |
|:-:|:-:|:-:|:-:|:-:|:-:|
| 예1 | (4, 4) | (3, 3) | 1 | 1 | (4 + 2 − 3)/1 + 1 = 4 → (4, 4) |
| 예2 | (7, 7) | (3, 3) | 0 | 2 | ( 　 , 　 ) |
| 예3 | (28, 31) | (5, 5) | 2 | 3 | ( 　 , 　 ) |

In [ ]:
Ap = np.pad(A, 1)                     # 둘레를 0 으로 1칸 채우기
print('패딩한 입력 (6 × 6) =\n', Ap)
print('컨볼루션 결과 (4 × 4) =\n', conv2d(Ap, F))

def out_size(H, W, FH, FW, P, S):
    return (H + 2*P - FH) // S + 1, (W + 2*P - FW) // S + 1
for args in [(4, 4, 3, 3, 1, 1), (7, 7, 3, 3, 0, 2), (28, 31, 5, 5, 2, 3)]:
    print(f'입력 {args[:2]}, 필터 {args[2:4]}, 패딩 {args[4]}, 스트라이드 {args[5]} → 출력 {out_size(*args)}')

## 📖 개념 다지기 5. 필터는 무엇을 찾을까?

필터의 숫자 배치에 따라 찾는 특징이 달라집니다.

| 필터 | 모양 | 찾는 것 |
|:-:|:-:|:-|
| 세로선 필터 | ( 1 0 −1 / 1 0 −1 / 1 0 −1 ) | 왼쪽이 밝고 오른쪽이 어두운 **세로 경계** |
| 가로선 필터 | ( 1 1 1 / 0 0 0 / −1 −1 −1 ) | 위가 밝고 아래가 어두운 **가로 경계** |

CNN 은 이런 필터의 숫자(가중치)를 사람이 정하지 않고 **오차역전파로 스스로 학습** 합니다.

## 🎬 애니메이션 2. 손글씨 숫자에서 세로·가로 획 찾기

손글씨 숫자 사진 위로 세로선 필터와 가로선 필터가 지나가며 특징 맵을 만듭니다. (ReLU 적용, 밝을수록 그 특징이 강함)

In [ ]:
#@title 🎬 애니메이션 2 실행 (필터가 찾은 특징)
from sklearn.datasets import load_digits
digit = load_digits().images[7]                       # 8 × 8 손글씨 숫자
img = np.kron(digit, np.ones((3, 3))) / 16            # 24 × 24 로 키우기
Fv = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]])
Fh = np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]])
Mv = np.maximum(0, conv2d(img, Fv)); Mh = np.maximum(0, conv2d(img, Fh))
n = Mv.shape[0]
order = [(i, j) for i in range(0, n, 2) for j in range(0, n, 2)]
frames = list(range(0, len(order), 3)) + [len(order) - 1]*8
fig, axes = plt.subplots(1, 3, figsize=(13, 4.6))
def draw(f):
    k = frames[f]; i, j = order[k]
    for ax in axes: ax.clear(); ax.axis('off')
    axes[0].imshow(img, cmap='gray'); axes[0].add_patch(plt.Rectangle((j - .5, i - .5), 3, 3, fill=False, ec='red', lw=2.5))
    axes[0].set_title('입력 사진 (빨간 칸 = 필터 위치)', fontsize=12)
    for ax, Mx, name in [(axes[1], Mv, '세로선 필터'), (axes[2], Mh, '가로선 필터')]:
        show = np.full_like(Mx, np.nan)
        for (a, b) in order[:k+1]: show[a:a+2, b:b+2] = Mx[a:a+2, b:b+2]
        ax.imshow(show, cmap='magma', vmin=0, vmax=max(Mv.max(), Mh.max()))
        ax.set_title(f'{name}의 특징 맵', fontsize=12)
anim = animation.FuncAnimation(fig, draw, frames=len(frames), interval=120)
plt.close(fig)
show_anim(anim)

> **[관찰 1]** 세로선 필터의 특징 맵에서는 숫자의 ( 세로 / 가로 ) 획 부분이 밝게 나타난다. 가로선 필터에서는 ( 　　 ) 획이 밝다.

## 🎬 애니메이션 3. conv → ReLU → max pooling 한 번에 보기

In [ ]:
#@title 🎬 애니메이션 3 실행 (CNN 한 층의 흐름)
C1 = conv2d(img, Fv); R1 = np.maximum(0, C1); P1 = max_pool(R1, 2)
stages = [(img, '입력 24 × 24', 'gray'), (C1, f'conv 결과 {C1.shape[0]} × {C1.shape[1]}', 'coolwarm'),
          (R1, 'ReLU (음수 → 0)', 'magma'), (P1, f'max pooling {P1.shape[0]} × {P1.shape[1]}', 'magma')]
frames = [k for k in range(4) for _ in range(8)] + [3]*6
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
def draw(f):
    k = frames[f]
    for i, ax in enumerate(axes):
        ax.clear(); ax.axis('off')
        if i <= k:
            M, t, cm = stages[i]
            if cm == 'coolwarm':
                m = np.abs(M).max(); ax.imshow(M, cmap=cm, vmin=-m, vmax=m)
            else:
                ax.imshow(M, cmap=cm)
            ax.set_title(t, fontsize=12)
anim = animation.FuncAnimation(fig, draw, frames=len(frames), interval=150)
plt.close(fig)
show_anim(anim)

> 💡 크기는 24 × 24 → 22 × 22 → 11 × 11 로 줄어들었지만, **세로 획이 있는 곳** 이라는 핵심 정보는 남아 있습니다. CNN 은 이런 층을 여러 번 쌓아 점점 복잡한 특징(획 → 곡선 → 숫자 모양)을 찾습니다.

## ✏️ 연습 문제

> **[문제 1]** 입력 ( 1 0 1 / 0 1 0 / 1 0 1 ), 필터 ( 1 0 / 0 1 ), 스트라이드 1 의 컨볼루션 결과를 손으로 구하세요.  ( 　 　 ) / ( 　 　 )

> **[문제 2]** MNIST 사진 (28, 28) 에 필터 (3, 3), 패딩 1, 스트라이드 1 로 컨볼루션한 뒤 2 × 2 max pooling 을 하면 크기는? ( 　 , 　 )

코드로 확인하세요.

In [ ]:
P = np.array([[1, 0, 1], [0, 1, 0], [1, 0, 1]])
K = np.array([[1, 0], [0, 1]])
print('문제 1 :\n', conv2d(______, ______))
h, w = out_size(28, 28, 3, 3, ______, 1)
print('문제 2 : conv 후', (h, w), '→ pooling 후', (h // 2, w // 2))

In [ ]:
#@title 🔑 정답 확인 { display-mode: "form" }
print('문제 1 : [[2, 0], [0, 2]]  (1×1 + 0×0 + 0×0 + 1×1 = 2 ...)')
print('문제 2 : conv 후 (28, 28) → pooling 후 (14, 14)   (빈칸: P, K, 1)')

## 📌 정리

| 핵심 | 내용 |
|:---|:---|
| 컨볼루션 | 필터를 옮기며 같은 자리끼리 곱해서 더하기 → 특징 맵 |
| ReLU | max(0, x) |
| 풀링 | 구역을 최댓값 등으로 압축 (max pooling) |
| 패딩 | 둘레를 0 으로 채워 크기 유지 |
| 출력 크기 | OH = (H + 2P − FH)/S + 1 |
| 학습 | 필터의 숫자도 W 처럼 오차역전파로 학습된다 |

### 🏆 도전 과제
사진은 픽셀의 **위치(공간)** 가 중요했습니다. 그렇다면 문장 "나는 학교에 간다" 처럼 **순서(시간)** 가 중요한 데이터는 어떻게 다룰까요? → 다음 시간: **RNN (순환 신경망)**

---

[⬅ 이전 : 제 13편 오차역전파 (2) - 은닉층 공식과 MNIST 고속 학습](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/13_backprop_2.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/00_index.ipynb)　|　[다음 : 제 15편 RNN - 순서를 기억하는 신경망 ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/15_rnn.ipynb)